# Project 2: Identify and Classify Suspicious Activity Within URLs.

#### Student Name: Isaac Kone
#### Student ID: 501280479 

## Introduction: 

A URL (Uniform Resource Locator) can be thought of as the digital equivalent of a physical street address. It is a unique identifier that enables us to pinpoint and access specific resources on the internet. In its composition, a URL typically includes a protocol (such as http or https), a domain name (e.g., www.example.com), a distinct file path (e.g., /folder/file.html), and occasionally, additional query parameters that deliver supplementary information for the server to handle the request effectively.

Threat actors will use URLs as a vector for cyberattacks. Understanding the structure and patterns of URLs is an essential component of cyber threat intelligence, as this knowledge can be use to identify and counter potential threats more effectively.

Here are some of the most common URL-based cyberattacks:

 - **SQL Injection:** Cyber adversaries tweak URLs, adding malicious SQL queries in query parameters or input fields. If a web app doesn't do its due diligence in validating or sanitizing these inputs, the attacker gets a free pass into the database, potentially swiping sensitive data or running harmful commands.

 - **Cross-Site Scripting (XSS):** Attackers can sneak malicious scripts into URLs, which the web app unwittingly executes. These scripts can do anything from stealing user data to altering web content or spreading malware.

 - **Directory Traversal:** By fiddling with URL paths, attackers can sidestep security barriers and access restricted directories and files on the webserver, potentially unearthing sensitive data or configuration details.

 - **Forceful Browsing:** Threat actors don't always need fancy tactics. Sometimes they can simply use URLs to barge directly into unprotected resources, sidestepping any access controls. They might try various URLs or employ automated tools to uncover sensitive pages or files.

 - **Cross-Site Request Forgery (CSRF):** This is a bit of a con job. Attackers trick users into performing actions on a web app they didn't intend to, by hiding malicious URLs in emails or web pages. When the user clicks on the URL, the attacker can impersonate them and carry out actions without the user even knowing it.

 - **Distributed Denial of Service (DDoS):** In this assault, threat actors commandeer multiple compromised devices to inundate a webserver with a torrent of requests via URLs, overloading it and causing disruptions.

 - **Exploiting Vulnerabilities:** Some attackers exploit known weak spots in the webserver or web app software. They use URLs for this, often relying on automated tools to scan for susceptible servers and try to gain unauthorized access or execute malicious commands.

## Your Mission

For this project, your mission is to design a Python program capable of examining the provided honeypot (a decoy system used to trap cyber adversaries) log and pinpointing whether a URL could potentially be part of a cyberattack. What's more, your program should be able to categorize the suspected attack type.

You'll need to use regular expressions (patterns used to match character combinations in strings) to trawl through the log and identify signs of potential cyber threats like SQL Injection, Cross-Site Scripting, and even PHP or Windows-related attacks.

Your program should add the type of attack(s) identified to the honeypot log - some URLs may contain one or more different types of attacks. And hey, while you're welcome to seek guidance from ChatGPT, remember that the code and regular expression generated might not always work as expected or give you the results you're not looking for.

## Objective Statement:

The objective of this project is to examine honeypot data for evidence of malicious activity by detecting patterns in the URLs associated with these activities.

This research is significant because it can help in identifying potential cyber threats and vulnerabilities within the system, allowing for the timely implementation of protective measures. By understanding the types and patterns of attacks, organizations can better secure their networks, protect sensitive data, and ensure the integrity and availability of their services.

Additionally, this project will provide you with a diverse array of valuable skills and knowledge that can be applied to multiple disciplines and industries, especially within the realm of cybersecurity.

Upon completion of Project 1: Identify and Classify Suspicious Activity Within URLs, you can expect to acquire the following skills and benefits:

 - **Develop programming skills:** You will hone your Python programming abilities by writing a program to read and analyze files. This experience will enable you to become proficient in handling diverse data formats and managing file I/O operations.

 - **Master regular expressions:** You will learn to employ regular expressions for detecting specific patterns within the data. This skill is invaluable for numerous programming and data analysis tasks, facilitating efficient pattern matching and data extraction.

 - **Expand cybersecurity knowledge:** By scrutinizing honeypot data and pinpointing potential security breaches, you will acquire an understanding of various cyber attack types, such as SQL Injection, Cross-Site Scripting, and more. This expertise is crucial for those pursuing careers in cybersecurity or related fields.

 - **Cultivate critical thinking and problem-solving abilities:** You will be required to examine intricate datasets and engage in critical thinking to discern suspicious activity patterns. This process will foster robust problem-solving competencies and equip you to confront real-world challenges.

 - **Enhance data analysis proficiency:** You will gain experience in data analysis through processing and interpreting honeypot data. This skill is essential across various fields, as the capacity to analyze and comprehend data underpins informed decision-making.

 - **Acquire understanding of how intrusion detection and prevention systems work:** By analyzing honeypot data, you will become familiar with intrusion detection and prevention systems, which can aid in the creation and refinement of such systems in the future.

 - **Strengthen communication and collaboration skills:** You may collaborate with peers or present your findings to stakeholders, thereby fostering essential communication and teamwork abilities.

## Project 2: Due July 18, 2025 (18:00).

### Start Program Code Here

#### Libraries used

This section loads all required libraries.

In [1]:
#!pipx install joblib
#!pipx install tqdm
import re
import pandas as pd
from urllib.parse import unquote
from html import unescape
from tqdm import tqdm #necessary for helpful progress bar/debugging when working with large files
from IPython.display import display, Image
from tabulate import tabulate
from joblib import Parallel, delayed
import multiprocessing


#### Set Data Path:

In [2]:
# Set paths
main_dir = 'D:/'
data_dir = main_dir +'Python Data/'
output_dir = main_dir +'Reports/'

# Set working directory
os.chdir(main_dir)

# Remove annoying error
pd.options.mode.chained_assignment = None  # default='warn'
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

NameError: name 'os' is not defined

#### Regular Expressions (regex):

This section contains the regex statements that will be used to locate suspicious activity

In [3]:
tqdm.pandas() # useful for progress bar when working with large files

#in effort to speed up performance we want to precompile all of our regex
KEYWORDS = [
    "SELECT", "INSERT", "UPDATE", "DELETE", "DROP", "UNION", "FROM", "WHERE", "AND", "OR",
    "NOT", "LIKE", "JOIN", "ORDER", "GROUP", "BY", "HAVING", "TABLE", "CREATE", "ALTER"
]

SQLI_KEYWORDS = re.compile(r'\b(?:' + '|'.join(map(re.escape, KEYWORDS)) + r')\b', re.IGNORECASE)
SPECIALS = re.compile(r"|".join(map(re.escape, ["'", '"', ';', '*', '/*', '--', '#', ',', '//'])))
URL_ENC = re.compile(r"/.*%[a-fA-F0-9]{2}.*?\b")
HEX = re.compile(r"0x[a-fA-F0-9]+")
TAUTOLOGY = re.compile(r"(.+)=(?=\1)")

# Individual precompiled patterns
SCRIPT_TAG_PATTERN = re.compile(r'<\s*script.*?>.*?<\s*/\s*script\s*>', re.IGNORECASE | re.DOTALL)
ON_EVENT_PATTERN = re.compile(r'on\w+\s*=', re.IGNORECASE)
JAVASCRIPT_SCHEME_PATTERN = re.compile(r'javascript\s*:', re.IGNORECASE)
IMG_SRC_JS_PATTERN = re.compile(r'<\s*img[^>]*\s+src\s*=\s*["\']?javascript:', re.IGNORECASE)
IFRAME_TAG_PATTERN = re.compile(r'<\s*iframe.*?>', re.IGNORECASE)
ALERT_FUNC_PATTERN = re.compile(r'alert\s*\(', re.IGNORECASE)
EVAL_FUNC_PATTERN = re.compile(r'eval\s*\(', re.IGNORECASE)
DOCUMENT_COOKIE_PATTERN = re.compile(r'document\.cookie', re.IGNORECASE)

# Dictionary of patterns
XSS_PATTERNS = {
    'script_tag': SCRIPT_TAG_PATTERN,
    'on_event': ON_EVENT_PATTERN,
    'javascript_scheme': JAVASCRIPT_SCHEME_PATTERN,
    'img_src_js': IMG_SRC_JS_PATTERN,
    'iframe_tag': IFRAME_TAG_PATTERN,
    'alert_func': ALERT_FUNC_PATTERN,
    'eval_func': EVAL_FUNC_PATTERN,
    'document_cookie': DOCUMENT_COOKIE_PATTERN,
}

DOTDOT_PATTERN = re.compile(r'(\.\./|\.\.\\)+')
ENCODED_DOTDOT_PATTERN = re.compile(r'%2e%2e(?:%2f|%5c)+', re.IGNORECASE)
DOUBLE_SLASH_PATTERN = re.compile(r'/{2,}|\\{2,}')
NULL_BYTE_PATTERN = re.compile(r'%00', re.IGNORECASE)
TRAVERSAL_PATTERNS = {
    'dotdot': DOTDOT_PATTERN,
    'encoded_dotdot': ENCODED_DOTDOT_PATTERN,
    'double_slash': DOUBLE_SLASH_PATTERN,
    'null_byte': NULL_BYTE_PATTERN,
}

ETC_PASSWD_PATTERN = re.compile(r'/etc/passwd', re.IGNORECASE)
BASH_HISTORY_PATTERN = re.compile(r'\.bash_history', re.IGNORECASE)
ENV_PATTERN = re.compile(r'\.env', re.IGNORECASE)
COMMAND_INJECTION_PATTERN = re.compile(r'(\||;|&&|\$\()')
EXEC_CALLS_PATTERN = re.compile(r'(eval|exec|system|popen)\s*\(', re.IGNORECASE)
CVE_PATH_PATTERN = re.compile(r'/(wp-admin|cgi-bin|solr|shell\.php|\.git)', re.IGNORECASE)
BASE64_PATTERN = re.compile(r'(base64_decode|[A-Za-z0-9+/]{20,}={0,2})', re.IGNORECASE)

EXPLOIT_PATTERNS = {
    'etc_passwd': ETC_PASSWD_PATTERN,
    'bash_history': BASH_HISTORY_PATTERN,
    'env_file': ENV_PATTERN,
    'command_injection': COMMAND_INJECTION_PATTERN,
    'exec_call': EXEC_CALLS_PATTERN,
    'cve_path': CVE_PATH_PATTERN,
    'base64_payload': BASE64_PATTERN,
}

FORCEFUL_BROWSING_PATTERN = re.compile(
    r'/(admin|config|debug|hidden|private|backup|\.git|\.env|\.DS_Store|web-inf|\.htaccess)', 
    re.IGNORECASE
)


#### Functions:

In [4]:
def parallel_detect_sqli(urls: pd.Series, n_jobs=multiprocessing.cpu_count()) -> pd.Series:
    """Gives us a big performance boost"""
    results = Parallel(n_jobs=n_jobs)('section_label
        delayed(detect_sqli)(url) for url in tqdm(urls)
    )
    return pd.Series(results, index=urls.index)

def detect_sqli(url: str) -> dict:
    """Returns a dict of all the detected sqli in the current url"""

    #rapid search for any matches set true if found and categorizes accordingly
    flags = {
        'hex': bool(HEX.search(url)),
        'taut': bool(TAUTOLOGY.search(url)),
        'kw': bool(SQLI_KEYWORDS.search(url)),
        'kw_enc': False  #encoding/decoding operations proved to be slow so it is processed separately 
    }

    if URL_ENC.search(url):
        try:
            decoded = unquote(url)
            if SQLI_KEYWORDS.search(decoded):
                flags['kw_enc'] = True
        except Exception:
            pass

    return flags

def detect_csrf(request: str) -> dict:
    """Parses a request string and returns CSRF-related flags."""

    #print(f"request: {request}")
    # Split lines and parse
    words = request.strip().split()
    #print(f"lines: {words}")
    method, url =words[0].strip(),words[1].strip()  
    headers = {}

    # Try to recover key-value headers from broken space-delimited format
    for i in range(2, len(words) - 1):
        if ':' in words[i]:
            key = words[i].rstrip(':').lower()
            value = words[i + 1]
            headers[key] = value

    #we flag any state changing request
    state_changing = method in ['POST', 'PUT', 'DELETE', 'PATCH']

    # Token check
    has_csrf_token = any(k in headers for k in ['x-csrf-token', 'csrf-token', 'x-xsrf-token'])
    has_origin = 'origin' in headers
    has_referer = 'referer' in headers

    # If both 'host' and 'referer' are present, and the domain in the referer does not match the host header, flag as suspicious
    suspicious_referer = False
    if has_referer and 'host' in headers:
        referer_host_match = re.search(r'https?://([^/]+)', headers['referer'])
        if referer_host_match:
            referer_host = referer_host_match.group(1).lower()
            if referer_host != headers['host'].lower():
                suspicious_referer = True

    return {
        'state_changing': state_changing,
        'missing_csrf_token': state_changing and not has_csrf_token,
        'missing_origin_or_referer': state_changing and not (has_origin or has_referer),
        'referer_mismatch': state_changing and suspicious_referer,
        'csrf_suspected': (
            state_changing and (
                not has_csrf_token or
                not (has_origin or has_referer) or
                suspicious_referer
            )
        ),
        'url': url
    }

def detect_xss(url: str) -> dict:
    """Scans a URL for common XSS patterns and returns matching flags as a dictionary"""
    try:
        url = unescape(unquote(url))  # decode %xx and HTML entities
    except Exception:
        pass  # fail-safe fallback

    #if we find any of our xss patterns we set the type of xss to true
    return {name: bool(pattern.search(url)) for name, pattern in XSS_PATTERNS.items()}

def detect_traversal(url: str) -> dict:
    """Detects directory traversal attempts using dot-dot, encoded paths, or null bytes."""
    try:
        url = unquote(url)
    except Exception:
        pass

    #if we find any of our dir traversal patterns we set the type of dir traversal to true
    return {name: bool(pattern.search(url)) for name, pattern in TRAVERSAL_PATTERNS.items()}

def detect_exploit(url: str) -> dict:
    """Scans input for exploit patterns like RCE, LFI, or encoded payloads."""
    try:
        url = unquote(url)
    except Exception:
        pass
    return {name: bool(pattern.search(url)) for name, pattern in EXPLOIT_PATTERNS.items()}

def detect_forceful_browsing(url: str) -> dict:
    """Scans a URL for common patterns and returns matching flags as a dictionary"""
    try:
        url = unquote(url)
    except:
        pass
    return {'forceful_browsing': bool(FORCEFUL_BROWSING_PATTERN.search(url))}


def detect_ddos(df: pd.DataFrame, time_window='60s', threshold=50) -> pd.Series:
    """
    Returns a Series of dictionaries with 'ddos_flag' per row based on actual_ip and timestamp.
    """
    if 'datetime' not in df:
        df.loc[:, 'datetime'] = pd.to_datetime(df['timestamp'], errors='coerce')

    valid_df = df[df['datetime'].notna()].set_index('datetime')

    request_counts = (
        valid_df.groupby('actual_ip')['request_url']
                .rolling(time_window)
                .count()
                .reset_index(name='req_per_window')
    )

    ddos_ips = request_counts[request_counts['req_per_window'] >= threshold]['actual_ip'].unique()

    return df['actual_ip'].map(lambda ip: {'ddos_flag': ip in ddos_ips})

def report(df: pd.DataFrame, *all_findings: pd.Series) -> (pd.DataFrame, pd.DataFrame):
    """
    Generates a detailed and summarized report of threat detections.
    - Detailed: Original input + flag columns + labels.
    - Summarized: 'ident' + unique labels per row.
    """
    # Merge all findings
    detection_df = pd.concat([pd.DataFrame(f.tolist()) for f in all_findings], axis=1)
    detection_df = detection_df.loc[:, ~detection_df.columns.duplicated()]

    combined = pd.concat([df.reset_index(drop=True), detection_df.reset_index(drop=True)], axis=1)

    # Threat mapping
    threat_map = {
        'csrf_suspected': 'CSRF',
        'ddos_flag': 'DDOS',
        'forceful_browsing': 'FB',
        'hex': 'SQLi',
        'taut': 'SQLi',
        'kw': 'SQLi',
        'kw_enc': 'SQLi',
        'script_tag': 'XSS',
        'on_event': 'XSS',
        'javascript_scheme': 'XSS',
        'img_src_js': 'XSS',
        'iframe_tag': 'XSS',
        'alert_func': 'XSS',
        'eval_func': 'XSS',
        'document_cookie': 'XSS',
    }

    # Build threat label per row
    def extract_labels(row):
        """Extracts and concatenates threat labels from active detection flags in a DataFrame row."""
        labels = {label for col, label in threat_map.items() if row.get(col)}
        return " | ".join(sorted(labels)) if labels else None

    combined["threats_detected_label"] = combined.apply(extract_labels, axis=1)
    combined["threats_detected"] = combined["threats_detected_label"].notna().astype(int)

    # Detailed report includes all
    detailed = combined[combined["threats_detected"] > 0]

    # Summarized report includes ident + threat label
    summarized = detailed[["ident", "threats_detected_label"]].copy()

    return detailed, summarized

#### Load Data:
The dataset is from a honeypot: log_file.csv

In [5]:
def load_csv_with_progress(path: str, chunksize: int = 100_000) -> pd.DataFrame:
    """Reads a large TSV file in chunks with progress and returns a cleaned DataFrame."""
    display(f"[+] Reading CSV in chunks from: {path}")
    chunks = []
    total_lines = sum(1 for _ in open(path, 'r', encoding='utf8'))
    with tqdm(total=total_lines, desc="Loading CSV") as pbar:
        for chunk in pd.read_csv(path, sep='\t', encoding='utf8', dtype='string', chunksize=chunksize):
            chunks.append(chunk)
            pbar.update(len(chunk))
    df = pd.concat(chunks, ignore_index=True)
    display(f"[+] Finished reading CSV. Total rows: {len(df)}")
    return df.dropna(subset=['request_url'])

#### Data Wrangling:

This secion contains the required functions used to clean and prepare the data

In [6]:

if __name__ == "__main__":
    display("[*] Starting threat analysis pipeline...")

    df = load_csv_with_progress("./log_file_2.csv")

    display("[*] Sanitizing urls...")
    urls = df['request_url'].str.replace(SPECIALS, '', regex=True)
    
    display("[*] Sanitizing requests...")
    requests = df['request_raw'].str.replace(SPECIALS, '', regex=True)

'[*] Starting threat analysis pipeline...'

'[+] Reading CSV in chunks from: ./log_file_2.csv'

Loading CSV: 100%|█████████████████████████████████████████████████████████████████████▉| 851321/851322 [00:15<00:00, 54101.30it/s]


'[+] Finished reading CSV. Total rows: 851321'

'[*] Sanitizing urls...'

'[*] Sanitizing requests...'

#### Search for suspicious activity:

In [ ]:
    display("[*] Detecting SQLis...")
    sqli_findings = parallel_detect_sqli(urls)

    display("[*] Detecting XSS...")
    xss_findings = urls.progress_map(detect_xss)

    display("[*] Detecting Forceful Browsing...")
    fb_findings= urls.progress_map(detect_forceful_browsing)

    display("[*] Detecting CSRF...")
    csrf_findings = requests.progress_map(detect_csrf)
    #print(csrf_findings)

    display("[*] Detecting DDOS...")
    ddos_findings = detect_ddos(df[['timestamp', 'actual_ip', 'request_url']])
    #print(ddos_findings)

    display("[*] Reporting...")
    detailed, summarized = report(df, xss_findings, sqli_findings, csrf_findings, ddos_findings, fb_findings)


#### Save the results for further analysis:

In [ ]:

    display(f"[+] Done. {len(detailed)} threats detected.")
    detailed.to_csv("reports_detailed.csv", index=False)
    summarized.to_csv("reports_summary.csv", index=False)
    display("[+] Detailed report saved to reports_detailed.csv")
    display("[+] Summary report saved to reports_summary.csv")


## Observations, Analysis, and Conclusions

In [ ]:
Upon reviewing the threat detection results, one notable pattern is the disproportionately high number of requests flagged as Distributed Denial of Service (DDoS) attacks. This may be attributed to the sensitivity settings used in the detect_ddos function, which flags any IP that issues more than 50 requests within a 60-second window. Additionally, SQL injection detection sometimes produces false positives — for example, URLs like /admin/update trigger the kw_flag simply due to the presence of common SQL keywords like "UPDATE," even when the request is benign. Another point of interest is the filename column, which, although often overlooked, can be a useful indicator: it allows correlation with file-based threat intelligence services like VirusTotal, particularly if hashes or executable paths are captured.

In [ ]:
The threat detection logic leverages rule-based techniques, offering transparency and simplicity. The strength of this approach lies in its predictability: patterns are clearly defined, and detection is deterministic. However, its weaknesses become apparent in nuanced or context-rich environments. For example, the DDoS module, while effective for spotting traffic bursts, doesn't account for legitimate surges caused by API calls, search engine crawlers, or cron jobs. Similarly, the SQLi detection module relies on static keyword matching without parsing parameters or understanding request context, which can mislabel non-malicious admin endpoints as threats. The detection of cross-site scripting (XSS), traversal, and exploit attempts is more accurate due to stricter pattern specificity, but these modules also suffer from the same rule-based rigidity.

The utility of the filename column should not be underestimated. In environments where uploaded files or remote access to scripts are logged, this field could be cross-referenced with external reputation systems (e.g., VirusTotal, Hybrid Analysis) or used to calculate cryptographic hashes for malware classification. This makes it valuable not just for flagging known malicious scripts but also for enriching logs with threat intelligence data.

In [ ]:
The detection pipeline provides an effective foundation for large-scale log analysis and security triage. Its performance and modular structure make it suitable for batch processing and integration into a broader SIEM or threat hunting workflow. However, tuning is necessary — especially for DDoS and SQLi modules — to reduce noise and improve the signal-to-noise ratio. Enhancing the detection logic with context-aware features, adaptive baselines, and external intelligence lookups (e.g., via the filename column) will elevate the precision and practical value of the reports. With these refinements, the system can evolve from a pattern-matching engine into a more nuanced detection and response framework.

# This concludes Project 2.